In [1]:
import pickle

import numpy as np
from scipy import sparse

from implicit.als import AlternatingLeastSquares

c:\Users\Duy\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
interaction_matrix = sparse.load_npz(
    "../data/processed/interaction_matrix.npz"
).tocsr()

print(interaction_matrix.shape)

(1407579, 235061)


In [3]:
with open("../data/processed/user_encoder.pkl", "rb") as f:
    user_encoder = pickle.load(f)

with open("../data/processed/item_encoder.pkl", "rb") as f:
    item_encoder = pickle.load(f)

In [4]:
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.1,
    iterations=20,
    random_state=42
)

c:\Users\Duy\AppData\Local\Programs\Python\Python312\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


In [5]:
model.fit(interaction_matrix)

100%|██████████| 20/20 [02:14<00:00,  6.73s/it]


In [6]:
user_index = 100

ids, scores = model.recommend(
    userid=user_index,
    user_items=interaction_matrix[user_index],
    N=10,
    filter_already_liked_items=True
)

In [7]:
original_items = item_encoder.inverse_transform(ids)

print(original_items)

[389814 382885 248455    546 170390 242905 368403 445105 342530 291877]


In [8]:
import pandas as pd

products = pd.read_csv("../data/processed/products.csv")

products[
    products["itemid"].isin(original_items)
]

,itemid,categoryid
496,546,1349
152311,170390,411
217078,242905,1051
222014,248455,1349
260804,291877,14
306049,342530,984
329180,368403,196
342106,382885,1059
348273,389814,1393
397642,445105,230


In [9]:
from recommender_wrapper import Recommender

wrapper = Recommender(
    model,
    interaction_matrix,
    user_encoder,
    item_encoder
)

In [10]:
with open("../models/recommender.pkl", "wb") as f:
    pickle.dump(wrapper, f)

In [11]:
result = wrapper.recommend(
    visitor_id=257597,
    top_k=5
)

print(result)

[{'item_id': 441668, 'score': 0.0025549158453941345}, {'item_id': 153778, 'score': 0.002361794700846076}, {'item_id': 466385, 'score': 0.0020587274339050055}, {'item_id': 113535, 'score': 0.0018870162311941385}, {'item_id': 210087, 'score': 0.0017231832025572658}]
